# 25 · Load: industry jobs, pay, hiring and output (QCEW, CES, QWI, BEA) → `staging`

**Source cards:** `nysdol_qcew.md`, `bls_qcew_county.md`, `nysdol_ces.md`, `census_qwi.md`, `bea_county_gdp.md`. Explored in: notebooks 07, 08 and 09.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_nysdol__qcew_annual` | area (NYC or borough) × ownership × NAICS × year, 2019–2025 | 24,114 |
| `stg_bls__qcew_county` | borough × year × ownership × (total or sector) | 1,365 |
| `stg_nysdol__ces_monthly` | CES series × month, NYC, 2019 → latest | 9,660 |
| `stg_census__qwi` | borough × sector × age or education group × quarter, private, both sexes | 39,648 |
| `stg_bea__county_gdp` | borough × BEA line (total, private, 20 sectors) × year | 660 |

These are industry ("employers in this area") measures, kept separate from the occupation facts (brief S8). Each has its own level; they're never added together. Reading the two QWI state files takes a few minutes.

In [1]:
import re
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "25-load-industry.ipynb"
BOROUGHS = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan", "36081": "Queens", "36085": "Staten Island"}
COUNTY_FIPS = {"Bronx County": "36005", "Kings County": "36047", "New York County": "36061", "Queens County": "36081",
               "Richmond County": "36085"}
FIRST_YEAR = 2019
conn = connect()
summary = []

def digits(s: pd.Series, name: str) -> pd.Series:
    """Text -> Int64; blank -> NULL (counted); anything else that isn't an integer stops the notebook."""
    s = s.str.strip().str.replace(",", "", regex=False)
    blank = s == ""
    if blank.any():
        print(f"  {name}: blank -> NULL: {blank.sum():,}")
    assert s[~blank].str.fullmatch(r"-?\d+").all(), f"{name}: non-integer text {s[~blank & ~s.str.fullmatch(r'-?[0-9]+')].unique()[:5]}"
    return pd.to_numeric(s.mask(blank), errors="raise").astype("Int64")

## 1. NYS DOL QCEW (issues 1–7)

NYC appears three times with identical numbers (Labor Market Area, MSA, WIR): the Labor Market Area copy is kept, plus the five `County` rows. Suppressed cells are missing rows, so there's nothing to convert; absent means "not published", never 0. Range sectors are written as their first code (`31`, `44`, `48`); the industry dimension maps them. `Annual Average Salary` (derived) is not staged. NAICS 2022 detail is missing below retail and information from 2022, so those rows get `naics_detail_complete = false`.

In [2]:
qf = raw_files("nysdol_qcew_annual").iloc[-1]
q = pd.read_csv(qf.path, dtype=str, keep_default_na=False)
q.columns = [c.strip() for c in q.columns]
q = q.apply(lambda s: s.str.strip())
nyc = (q.Area == "New York City") & (q["Area Type"] == "Labor Market Area")
boro = (q["Area Type"] == "County") & q.Area.isin(COUNTY_FIPS)
yr = q.Year.astype(int) >= FIRST_YEAR
qq = q[(nyc | boro) & yr].copy()
log(f"keep NYC (Labor Market Area) + 5 borough counties, {FIRST_YEAR}+", len(q), len(qq))
qq["area_code"] = qq.Area.map(COUNTY_FIPS).fillna("NYC")
assert set(qq.NAICS.str.len()) <= {2, 3, 4}
qq["naics_level"] = qq.NAICS.map(lambda c: 0 if c == "00" else len(c)).astype("Int64")
qq["year"] = qq.Year.astype(int).astype("Int64")
qq["naics_detail_complete"] = ~(qq.NAICS.str[:2].isin(["44", "45", "51"]) & (qq.year >= 2022))
for src, dst in [("Establishments", "establishments"), ("Average Employment", "employment"), ("Total Wage", "total_wages")]:
    qq[dst] = digits(qq[src], dst)
print("Ownerships:", qq.Ownership.value_counts().to_dict())
stg_q = (qq.rename(columns={"Area": "area_name", "Ownership": "ownership", "NAICS": "naics_code", "NAICS Title": "naics_title"})
         [["area_code", "area_name", "ownership", "naics_code", "naics_level", "naics_title", "year", "establishments",
           "employment", "total_wages", "naics_detail_complete"]]
         .assign(source_file=qf.file).sort_values(["area_code", "ownership", "naics_code", "year"]).reset_index(drop=True))

# Tie-out: the five boroughs add to NYC within +-3 jobs (card §2)
priv = stg_q[(stg_q.ownership == "Private") & (stg_q.naics_code == "00")]
t = priv.pivot_table(index="year", columns=priv.area_code.eq("NYC").map({True: "NYC", False: "boroughs"}), values="employment", aggfunc="sum")
t["diff"] = t.boroughs - t.NYC
display(t.T); assert t["diff"].abs().max() <= 3
summary.append(stage(stg_q, "stg_nysdol__qcew_annual", ["area_code", "ownership", "naics_code", "year"], expected_rows=24_114,
                     source_key="nysdol_qcew_annual", notebook=NB, conn=conn,
                     comment="NYS DOL QCEW annual, NYC + boroughs, 2019+. Missing row = suppressed. Use 2-digit sectors in v1."))

  keep NYC (Labor Market Area) + 5 borough counties, 2019+: 1,163,386 -> 24,114 rows (-1,139,272)
Ownerships: {'Private': 11873, 'Total, All Ownerships': 10327, 'Federal Government': 973, 'State Government': 572, 'Total, All Government': 199, 'Local Government': 170}


year,2019,2020,2021,2022,2023,2024,2025
area_code,,,,,,,
NYC,3916345,3421539,3506836,3793900,3928359,4063618,4074085
boroughs,3916346,3421539,3506837,3793901,3928360,4063618,4074085
diff,1,0,1,1,1,0,0


  stg_nysdol__qcew_annual: 24,114 rows x 12 cols, key ['area_code', 'ownership', 'naics_code', 'year'] unique | local data/interim/staging/stg_nysdol__qcew_annual.parquet | DB: 24114


## 2. BLS QCEW county files: government by sector (nb 09)

NYS DOL's extract has no sector detail for government; BLS's county files do. Kept: `agglvl_code` 71 (county × ownership, all industries) and 74 (county × ownership × sector). Suppressed cells show 0 jobs with `disclosure_code = N`, so they become NULL with `disclosed = false`.

In [3]:
bf = raw_files("bls_qcew_county")
parts = []
for f, p in zip(bf.file, bf.path):
    d = pd.read_csv(p, dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
    parts.append(d.assign(source_file=f))
b = pd.concat(parts, ignore_index=True)
assert set(b.size_code) == {"0"} and set(b.qtr) == {"A"}
bb = b[b.agglvl_code.isin(["71", "74"])].copy()
log("keep agglvl 71 (county x ownership) and 74 (x sector)", len(b), len(bb))
bb["disclosed"] = bb.disclosure_code.ne("N")
print("Disclosure codes:", bb.disclosure_code.replace("", "(blank)").value_counts().to_dict())
for src, dst in [("annual_avg_estabs", "estabs"), ("annual_avg_emplvl", "emp"), ("total_annual_wages", "total_wages"),
                 ("avg_annual_pay", "avg_annual_pay")]:
    v = digits(bb[src], dst)
    bb[dst] = v.mask(~bb.disclosed)                       # suppressed: 0 in the file -> NULL
print(f"Suppressed rows -> NULL measures: {(~bb.disclosed).sum()}")
bb["year"] = bb.year.astype(int).astype("Int64")
stg_b = (bb.rename(columns={"area_fips": "county_fips"})
         [["county_fips", "year", "own_code", "industry_code", "agglvl_code", "estabs", "emp", "total_wages",
           "avg_annual_pay", "disclosed", "source_file"]]
         .sort_values(["county_fips", "year", "own_code", "industry_code"]).reset_index(drop=True))
assert set(stg_b.county_fips) == set(BOROUGHS)
summary.append(stage(stg_b, "stg_bls__qcew_county", ["county_fips", "year", "own_code", "industry_code"], expected_rows=1_365,
                     source_key="bls_qcew_county", notebook=NB, conn=conn,
                     comment="BLS QCEW county annual averages, NYC boroughs, ownership x (total | sector). Suppressed = NULL, disclosed = false."))

  keep agglvl 71 (county x ownership) and 74 (x sector): 63,485 -> 1,365 rows (-62,120)
Disclosure codes: {'(blank)': 1101, 'N': 264}
Suppressed rows -> NULL measures: 264
  stg_bls__qcew_county: 1,365 rows x 11 cols, key ['county_fips', 'year', 'own_code', 'industry_code'] unique | local data/interim/staging/stg_bls__qcew_county.parquet | DB: 1365


## 3. CES monthly, NYC (issues 8–11)

`Area = 93561` (NYC). Months 1–12 only (`Month = 0` is the annual average, rebuilt in core). Rounded to the hundred; the latest months are preliminary (revised each March), so the download date is stored as `vintage`.

In [4]:
cf = raw_files("nysdol_ces").iloc[-1]
c = pd.read_csv(cf.path, dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
cc = c[(c.Area == "93561") & (c.Month != "0") & (c.Year.astype(int) >= FIRST_YEAR)].copy()
log(f"keep NYC (93561), months 1-12, {FIRST_YEAR}+", len(c), len(cc))
cc["employment"] = digits(cc["Current Employment"], "employment")
cc["year"] = cc.Year.astype(int).astype("Int64")
cc["month"] = cc.Month.astype(int).astype("Int64")
assert cc.month.between(1, 12).all() and cc.Series.str.fullmatch(r"\d{8}").all()
cc["supersector_code"] = cc.Series.str[:2]
stg_c = (cc.rename(columns={"Series": "series_code", "Title": "series_title"})
         [["series_code", "series_title", "supersector_code", "year", "month", "employment"]]
         .assign(vintage=cf.downloaded_utc[:10], source_file=cf.file)
         .sort_values(["series_code", "year", "month"]).reset_index(drop=True))
print(f"{stg_c.series_code.nunique()} series, {stg_c.year.min()}-01 -> {stg_c.year.max()}-{stg_c[stg_c.year == stg_c.year.max()].month.max():02d}")
summary.append(stage(stg_c, "stg_nysdol__ces_monthly", ["series_code", "year", "month"], expected_rows=9_660,
                     source_key="nysdol_ces", notebook=NB, conn=conn,
                     comment="NYS DOL CES monthly jobs, NYC (93561), 2019+. CES codes, not NAICS. Latest months preliminary."))

  keep NYC (93561), months 1-12, 2019+: 498,400 -> 9,660 rows (-488,740)
105 series, 2019-01 -> 2026-08
  stg_nysdol__ces_monthly: 9,660 rows x 8 cols, key ['series_code', 'year', 'month'] unique | local data/interim/staging/stg_nysdol__ces_monthly.parquet | DB: 9660


## 4. QWI: hiring by borough × sector × age or education (nb 08 issues 1–9)

Private ownership (the `op` files), both sexes (`sex = 0`), 2019 → latest. Age and education come in separate files and are never crossed, so they're stacked with a `breakdown` column; the all-groups total (`A00`) is kept once (from the age file), and `E0` is dropped. A value is kept only when its status flag is released (1, 6, 9, 10); otherwise NULL, with the flag kept.

In [5]:
qwf = raw_files("census_qwi")
version = Path(qwf[qwf.file.str.endswith("version_qwi.txt")].path.iloc[0]).read_text()
release = re.search(r"R\d{4}Q\d", version).group(0)
print("QWI release:", release)
VARS = {"Emp": "emp_begin", "EmpS": "emp_stable", "HirA": "hires_all", "HirN": "hires_new", "Sep": "separations",
        "EarnS": "earn_stable_monthly", "EarnHirNS": "earn_new_hire_stable_monthly", "FrmJbC": "job_creation"}
DIMS = ["geography", "industry", "ind_level", "ownercode", "sex", "agegrp", "education", "year", "quarter"]
USE = DIMS + list(VARS) + ["s" + v for v in VARS]

def borough_rows(path: Path) -> pd.DataFrame:
    n, keep = 0, []
    for ch in pd.read_csv(path, dtype=str, keep_default_na=False, usecols=USE, chunksize=500_000):
        n += len(ch)
        keep.append(ch[ch.geography.isin(BOROUGHS) & (ch.sex == "0") & (ch.year.astype(int) >= FIRST_YEAR)])
    out = pd.concat(keep, ignore_index=True)
    log(f"{path.name}: keep 5 boroughs, sex = 0, {FIRST_YEAR}+", n, len(out))
    return out

qa = borough_rows(qwf[qwf.file.str.contains("_sa_")].path.iloc[0]).assign(breakdown="age")
qe = borough_rows(qwf[qwf.file.str.contains("_se_")].path.iloc[0])
before = len(qe); qe = qe[qe.education != "E0"].assign(breakdown="education")
log("education file: drop E0 (all groups, already in the age file)", before, len(qe))
w = pd.concat([qa, qe], ignore_index=True)
w["group_code"] = w.agegrp.where(w.breakdown == "age", w.education)
print("Ownership codes:", w.ownercode.unique(), "| industry levels:", w.ind_level.value_counts().to_dict())

RELEASED = {"1", "6", "9", "10"}
flags = []
for v, dst in VARS.items():
    ok = w["s" + v].isin(RELEASED)
    flags.append({"measure": dst, "released": int(ok.sum()), "not released -> NULL": int((~ok).sum()),
                  "distorted (9)": int((w["s" + v] == "9").sum())})
    w[dst] = pd.to_numeric(w[v].where(ok).replace("", pd.NA), errors="raise").astype("Int64")
    w[dst + "_flag"] = pd.to_numeric(w["s" + v], errors="raise").astype("Int64")
display(pd.DataFrame(flags).set_index("measure").T)

w["year"] = w.year.astype(int).astype("Int64")
w["quarter"] = w.quarter.astype(int).astype("Int64")
last = (w.year * 10 + w.quarter).max()
w["quarter_complete"] = (w.year * 10 + w.quarter) < last
stg_w = (w.rename(columns={"geography": "county_fips", "industry": "naics_sector"})
         [["county_fips", "naics_sector", "breakdown", "group_code", "year", "quarter", *VARS.values(),
           *[d + "_flag" for d in VARS.values()], "quarter_complete"]]
         .assign(release=release).sort_values(["county_fips", "naics_sector", "breakdown", "group_code", "year", "quarter"])
         .reset_index(drop=True))
print(f"Quarters: {stg_w.year.min()}Q1 -> {last // 10}Q{last % 10} (latest marked incomplete)")
summary.append(stage(stg_w, "stg_census__qwi", ["county_fips", "naics_sector", "breakdown", "group_code", "year", "quarter"],
                     expected_rows=39_648, source_key="census_qwi", notebook=NB, conn=conn,
                     comment="Census QWI, NYC boroughs, private, both sexes, by age or education (never crossed). Noise-infused, not seasonally adjusted."))

QWI release: R2026Q3


  qwi_ny_sa_f_gc_ns_op_u.csv.gz: keep 5 boroughs, sex = 0, 2019+: 3,553,254 -> 25,488 rows (-3,527,766)


  qwi_ny_se_f_gc_ns_op_u.csv.gz: keep 5 boroughs, sex = 0, 2019+: 2,368,836 -> 16,992 rows (-2,351,844)
  education file: drop E0 (all groups, already in the age file): 16,992 -> 14,160 rows (-2,832)
Ownership codes: <ArrowStringArray>
['A05']
Length: 1, dtype: str | industry levels: {'S': 37688, 'A': 1960}


measure,emp_begin,emp_stable,hires_all,hires_new,separations,earn_stable_monthly,earn_new_hire_stable_monthly,job_creation
released,38559,37147,38368,38278,37037,36615,35039,37104
not released -> NULL,1089,2501,1280,1370,2611,3033,4609,2544
distorted (9),0,0,0,0,0,0,0,0


Quarters: 2019Q1 -> 2025Q4 (latest marked incomplete)

  stg_census__qwi: 39,648 rows x 24 cols, key ['county_fips', 'naics_sector', 'breakdown', 'group_code', 'year', 'quarter'] unique | local data/interim/staging/stg_census__qwi.parquet | DB: 39648


## 5. BEA county GDP (issues 12–16)

Years wide → one row per year; quoted, padded `GeoFIPS`; 4 footer rows. `(D)` = withheld → NULL with `is_withheld`. Kept lines: total (1), private (2) and the 20 NAICS sectors; combined lines (e.g. "Trade", "Professional and business services") double-count and are dropped. Current dollars (CAGDP2) and chained 2017 dollars (CAGDP9) share the key and are joined. The release date comes from the footer.

In [6]:
gf = raw_files("bea_county_gdp")
YEARS = [str(y) for y in range(FIRST_YEAR, 2025)]

def read_bea(table: str):
    p = gf[gf.file.str.contains(table)].path.iloc[0]
    z = zipfile.ZipFile(p)
    d = pd.read_csv(z.open(next(n for n in z.namelist() if n.startswith(f"{table}_NY_"))), dtype=str,
                    keep_default_na=False, encoding="latin-1")
    is_row = d.LineCode.str.strip().str.fullmatch(r"\d+")
    footer = " ".join(d.loc[~is_row, "GeoFIPS"])
    d = d[is_row].apply(lambda s: s.str.strip())
    d["county_fips"] = d.GeoFIPS.str.replace('"', "").str.strip()
    d = d[d.county_fips.isin(BOROUGHS)]
    keep = d.LineCode.isin(["1", "2"]) | d.IndustryClassification.str.fullmatch(r"\d\d(-\d\d)?")
    d = d[keep]
    long = d.melt(id_vars=["county_fips", "GeoName", "LineCode", "IndustryClassification", "Description"], value_vars=YEARS,
                  var_name="year", value_name="value")
    return long, footer

g2, footer = read_bea("CAGDP2")
g9, _ = read_bea("CAGDP9")
release = pd.to_datetime(re.search(r"Last updated: (\w+ \d+, \d{4})", footer).group(1)).date().isoformat()
print("BEA release:", release, "| lines kept per borough:", g2.groupby("county_fips").LineCode.nunique().unique())
g = g2.merge(g9[["county_fips", "LineCode", "year", "value"]], on=["county_fips", "LineCode", "year"], suffixes=("_cur", "_real"),
             how="outer", indicator=True)
assert (g._merge == "both").all()
print("Withheld '(D)' cells:", {c: int((g[c] == "(D)").sum()) for c in ["value_cur", "value_real"]},
      "| other symbols:", sorted(set(g.value_cur[~g.value_cur.str.fullmatch(r"\d+")]) | set(g.value_real[~g.value_real.str.fullmatch(r"\d+")])))
g["is_withheld"] = g.value_cur.eq("(D)") | g.value_real.eq("(D)")
for src, dst in [("value_cur", "gdp_current_thousands"), ("value_real", "gdp_real_chained2017_thousands")]:
    g[dst] = digits(g[src].replace("(D)", ""), dst)
stg_g = (g.rename(columns={"GeoName": "county_name", "LineCode": "line_code", "IndustryClassification": "naics_label",
                           "Description": "line_title"})
         .assign(year=lambda d: d.year.astype(int).astype("Int64"), line_code=lambda d: d.line_code.astype(int).astype("Int64"),
                 release=release, source_file=", ".join(gf.file))
         [["county_fips", "county_name", "line_code", "naics_label", "line_title", "year", "gdp_current_thousands",
           "gdp_real_chained2017_thousands", "is_withheld", "release", "source_file"]]
         .sort_values(["county_fips", "line_code", "year"]).reset_index(drop=True))
summary.append(stage(stg_g, "stg_bea__county_gdp", ["county_fips", "line_code", "year"], expected_rows=660,
                     source_key="bea_county_gdp", notebook=NB, conn=conn,
                     comment="BEA GDP by county (NYC boroughs), total + private + 20 sectors, current and chained 2017 $ (thousands). Context only."))

BEA release: 2026-02-05 | lines kept per borough: [22]
Withheld '(D)' cells: {'value_cur': 36, 'value_real': 36} | other symbols: ['(D)']
  gdp_current_thousands: blank -> NULL: 36
  gdp_real_chained2017_thousands: blank -> NULL: 36
  stg_bea__county_gdp: 660 rows x 11 cols, key ['county_fips', 'line_code', 'year'] unique | local data/interim/staging/stg_bea__county_gdp.parquet | DB: 660


## 6. Summary

In [7]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_nysdol__qcew_annual,24114,12,area_code + ownership + naics_code + year,nysdol_qcew_annual,25-load-industry.ipynb,24114,2026-10-14T21:12:34+00:00
1,stg_bls__qcew_county,1365,11,county_fips + year + own_code + industry_code,bls_qcew_county,25-load-industry.ipynb,1365,2026-10-14T21:12:35+00:00
2,stg_nysdol__ces_monthly,9660,8,series_code + year + month,nysdol_ces,25-load-industry.ipynb,9660,2026-10-14T21:12:36+00:00
3,stg_census__qwi,39648,24,county_fips + naics_sector + breakdown + group_code + year + quarter,census_qwi,25-load-industry.ipynb,39648,2026-10-14T21:12:54+00:00
4,stg_bea__county_gdp,660,11,county_fips + line_code + year,bea_county_gdp,25-load-industry.ipynb,660,2026-10-14T21:12:54+00:00
